### Building a RAG System with Langchain and ChromaDB
#### Introduction
Retrieval-Augmented Generation (RAG) is a powerful technique that combines the capabilities of large language models with external knowledge retirieval. This notebook will walk you through building a complete RAG system using:

- Langchain: A framework for developing applications powered by language models.
- ChromaDB: An open-source vector database for storing and retrieving embeddings.
- OpenAI or GEMINI(Google GenAI): For embeddings and language model(you can substitute with other providers as well)

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
## LAngchain imports
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_google_genai import GoogleGenerativeAIEmbeddings


## Vectorstores
from langchain_community.vectorstores import Chroma

## utility inputs
import numpy as np
from typing import List

C:\Users\mihir\AppData\Local\Temp\ipykernel_11660\1392439393.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


RAG (Retrieval-Augmented Generation) Architecture:

1. Document Loading: Load documents from various sources
2. Document Splitting: Break documents into smaller chunks
3. Embedding Generation: Convert chunks into vector representations
4. Vector Storage: Store embeddings in ChromaDB
5. Query Processing: Convert user query to embedding
6. Similarity Search: Find relevant chunks from vector store
7. Context Augmentation: Combine retrieved chunks with query
8. Response Generation: LLM generates answer using context

Benefits of RAG:
- Reduces hallucinations
- Provides up-to-date information
- Allows citing sources
- Works with domain-specific knowledge

## 1. Sample Data

In [3]:
## Create sample dcouments
sample_docs=["""
Machine Learning (ML) focuses on training statistical algorithms to identify patterns in historical data and make predictions or decisions without relying on explicitly programmed rules. It performs especially well on structured and tabular datasets where domain-relevant features are carefully engineered and selected beforehand. Common supervised and unsupervised approaches include linear and logistic regression, decision trees, random forests, gradient boosting, support vector machines, and k-means clustering. Organizations rely on classical ML for practical tasks such as credit scoring, fraud detection, customer churn prediction, demand forecasting, and recommendation engines, largely because these models are fast to train, require moderate data volumes, and remain relatively interpretable.""",

"""Deep Learning (DL) is a specialized branch of machine learning that uses multi-layer artificial neural networks to learn hierarchical representations directly from raw, unstructured data. Rather than depending on manual feature extraction, deep neural networks automatically discover low-level to high-level abstractions across successive layers using backpropagation and gradient-based optimization. Key architectures include convolutional neural networks for spatial and visual data, recurrent networks for sequential modeling, and transformers for parallel sequence processing at scale. DL powers breakthroughs in computer vision, speech recognition, autonomous systems, and generative media, though it typically demands large datasets and substantial GPU compute.""",

"""Natural Language Processing (NLP) enables computers to analyze, understand, and generate human language across both text and speech. Early NLP systems relied on handcrafted linguistic rules, tokenization, part-of-speech tagging, and classical statistical models such as TF-IDF paired with linear classifiers. Modern NLP has shifted toward deep learning—most notably pretrained transformer architectures and dense vector embeddings—that capture nuanced context, semantics, and long-range dependencies across sentences and documents. Today, NLP underpins machine translation, sentiment analysis, semantic search, document summarization, conversational assistants, and retrieval-augmented generation workflows."""]

sample_docs

['\nMachine Learning (ML) focuses on training statistical algorithms to identify patterns in historical data and make predictions or decisions without relying on explicitly programmed rules. It performs especially well on structured and tabular datasets where domain-relevant features are carefully engineered and selected beforehand. Common supervised and unsupervised approaches include linear and logistic regression, decision trees, random forests, gradient boosting, support vector machines, and k-means clustering. Organizations rely on classical ML for practical tasks such as credit scoring, fraud detection, customer churn prediction, demand forecasting, and recommendation engines, largely because these models are fast to train, require moderate data volumes, and remain relatively interpretable.',
 'Deep Learning (DL) is a specialized branch of machine learning that uses multi-layer artificial neural networks to learn hierarchical representations directly from raw, unstructured data. 

In [4]:
##save sample doc to file
for i, doc in enumerate(sample_docs):
    with open(f"doc_{i}.txt","w") as f:
        f.write(doc)


### 2. Document Loading

In [5]:
from langchain_community.document_loaders import DirectoryLoader , TextLoader

#load documents from directory
loader = DirectoryLoader(
    "data",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"autodetect_encoding": True}
)
documents=loader.load()
print(f"Loaded {len(documents)} documents")
print(f"\nFirst document preview:")
print(documents[0].page_content[:200] + "...")

Loaded 3 documents

First document preview:

Machine Learning (ML) focuses on training statistical algorithms to identify patterns in historical data and make predictions or decisions without relying on explicitly programmed rules. It performs ...


### 3. Document Splitting

In [6]:
#initialing text splitter
text_splitter= RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
    separators=[" "]
)
chunks=text_splitter.split_documents(documents)

print(f"Created {len(chunks)} chunks from {len(documents)} documents")
print(f"\nChunk example:")
print(f"Content: {chunks[0].page_content[:150]}...")
print(f"Metadata: {chunks[0].metadata}")

Created 6 chunks from 3 documents

Chunk example:
Content: Machine Learning (ML) focuses on training statistical algorithms to identify patterns in historical data and make predictions or decisions without rel...
Metadata: {'source': 'data\\doc_0.txt'}


In [7]:
chunks

[Document(metadata={'source': 'data\\doc_0.txt'}, page_content='Machine Learning (ML) focuses on training statistical algorithms to identify patterns in historical data and make predictions or decisions without relying on explicitly programmed rules. It performs especially well on structured and tabular datasets where domain-relevant features are carefully engineered and selected beforehand. Common supervised and unsupervised approaches include linear and logistic regression, decision trees, random forests, gradient boosting, support vector machines, and'),
 Document(metadata={'source': 'data\\doc_0.txt'}, page_content='gradient boosting, support vector machines, and k-means clustering. Organizations rely on classical ML for practical tasks such as credit scoring, fraud detection, customer churn prediction, demand forecasting, and recommendation engines, largely because these models are fast to train, require moderate data volumes, and remain relatively interpretable.'),
 Document(meta

### 4. Embedding models

### Initilize the ChromaDB Vector Store And Stores the chunks in vector Format

In [8]:
# Create a ChromaDB vector Store
persist_directory="./chroma_db"

#Initilaize ChromaDB with GEMINI embeddings
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2")

vectorstore=Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="rag_collection"
)

print(f"Vector store created with {vectorstore._collection.count()} vectors")
print(f"Persisted to: {persist_directory}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Vector store created with 6 vectors
Persisted to: ./chroma_db


### Test Similarity Search

In [9]:
query="what is NLP?"

similar_docs=vectorstore.similarity_search(query,k=3)
similar_docs

[Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP) enables computers to analyze, understand, and generate human language across both text and speech. Early NLP systems relied on handcrafted linguistic rules, tokenization, part-of-speech tagging, and classical statistical models such as TF-IDF paired with linear classifiers. Modern NLP has shifted toward deep learning—most notably pretrained transformer architectures and dense vector embeddings—that capture nuanced context, semantics, and long-range dependencies'),
 Document(metadata={'source': 'data\\doc_2.txt'}, page_content='context, semantics, and long-range dependencies across sentences and documents. Today, NLP underpins machine translation, sentiment analysis, semantic search, document summarization, conversational assistants, and retrieval-augmented generation workflows.'),
 Document(metadata={'source': 'data\\doc_1.txt'}, page_content='networks for spatial and visual data, recurren

In [10]:
print(f"Query: {query}")
print(f"\nTop {len(similar_docs)} similar chunks:")
for i, doc in enumerate(similar_docs):
    print(f"\n--- Chunk {i+1} ---")
    print(doc.page_content[:200] + "...")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}")

Query: what is NLP?

Top 3 similar chunks:

--- Chunk 1 ---
Natural Language Processing (NLP) enables computers to analyze, understand, and generate human language across both text and speech. Early NLP systems relied on handcrafted linguistic rules, tokenizat...
Source: data\doc_2.txt

--- Chunk 2 ---
context, semantics, and long-range dependencies across sentences and documents. Today, NLP underpins machine translation, sentiment analysis, semantic search, document summarization, conversational as...
Source: data\doc_2.txt

--- Chunk 3 ---
networks for spatial and visual data, recurrent networks for sequential modeling, and transformers for parallel sequence processing at scale. DL powers breakthroughs in computer vision, speech recogni...
Source: data\doc_1.txt


### Advance similarity search with score

In [11]:
result=vectorstore.similarity_search_with_score(query,k=3)

In [12]:
result

[(Document(metadata={'source': 'data\\doc_2.txt'}, page_content='Natural Language Processing (NLP) enables computers to analyze, understand, and generate human language across both text and speech. Early NLP systems relied on handcrafted linguistic rules, tokenization, part-of-speech tagging, and classical statistical models such as TF-IDF paired with linear classifiers. Modern NLP has shifted toward deep learning—most notably pretrained transformer architectures and dense vector embeddings—that capture nuanced context, semantics, and long-range dependencies'),
  0.5120623111724854),
 (Document(metadata={'source': 'data\\doc_2.txt'}, page_content='context, semantics, and long-range dependencies across sentences and documents. Today, NLP underpins machine translation, sentiment analysis, semantic search, document summarization, conversational assistants, and retrieval-augmented generation workflows.'),
  0.5501870512962341),
 (Document(metadata={'source': 'data\\doc_1.txt'}, page_conten

#### 2. Understanding Similarity Scores
The similarity score represents how closely related a document chunk is to your query. The scoring depends on the metric used:

ChromaDB default: Uses L2 distance (Euclidean distance)

- Lower scores = MORE similar (closer in vector space)
- Score of 0 = identical vectors
- Typical range: 0 to 2 (but can be higher)


Cosine similarity (if configured):

-  Higher scores = MORE similar
-  Range: -1 to 1 (1 being identical)

### Initialize LLM, RAG Chain, Prompt Template, Query the RAG system

In [16]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm=ChatGoogleGenerativeAI(
    model="gemini-3.5-flash"
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [17]:
test_response=llm.invoke("what is LLM?")
test_response

AIMessage(content=[{'type': 'text', 'text': '**LLM** stands for **Large Language Model**. \n\nIt is a type of Artificial Intelligence (AI) designed to understand, generate, and process human language. LLMs are the underlying technology behind popular AI chatbots like ChatGPT, Google Gemini, and Claude.\n\nHere is a breakdown of what the name means, how it works, and what it can do:\n\n---\n\n### 1. Breaking Down the Name\n* **Large:** Refers to two things:\n  1. The **massive amount of data** used to train it (hundreds of billions of words from books, articles, websites, and code).\n  2. The **size of the network** itself (often billions of parameters/connections that help it process information).\n* **Language:** It is built specifically to handle human communication (English, Spanish, Python code, etc.).\n* **Model:** It is a computer program trained to recognize patterns and make predictions based on data.\n\n---\n\n### 2. How Does an LLM Work?\nAt its core, an LLM works like a **hy

In [19]:
from langchain.chat_models.base import init_chat_model
llm=init_chat_model("google_genai:gemini-3.5-flash")
llm

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.4.0', 'langchain-google-genai': '4.4.0'}}, profile={'name': 'Gemini 3.5 Flash', 'release_date': '2026-05-19', 'last_updated': '2026-05-19', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True, 'reasoning_effort_levels': ['minimal', 'low', 'medium', 'high'], 'reasoning_effort_default': 'medium'}, google_api_key=SecretStr('**********'), model='gemini-3.5-flash', temperature=None, client=<google.genai.client.Client object at 0x000001C46DFB0740>, default_metadata=(), model_kwargs={})

In [21]:
llm.invoke("what is Desi Girl")

AIMessage(content=[{'type': 'text', 'text': 'The term **"Desi Girl"** has a few different meanings depending on whether you are talking about literal culture, pop culture, or social media trends. \n\nHere is a breakdown of what it means:\n\n---\n\n### 1. Literal Meaning & Cultural Context\n* **"Desi"** (derived from the Sanskrit word *desh*, meaning "country") refers to people, culture, and products from South Asia—primarily **India, Pakistan, and Bangladesh**.\n* A **"Desi Girl"** is simply a woman of South Asian heritage, whether she lives in South Asia or as part of the global diaspora (like in the US, UK, or Canada).\n\n---\n\n### 2. The Pop Culture Connection (Bollywood)\nThe term was popularized globally by the **2008 Bollywood film *Dostana***, which featured a hit song titled **"Desi Girl."** \n\n* **Priyanka Chopra:** Actress Priyanka Chopra starred in the song wearing a famous metallic saree. The song became a massive hit, and ever since, Priyanka Chopra has been nicknamed **

### Modern RAG Chain

In [28]:
from langchain_classic.chains import create_retrieval_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [26]:
### Convert Vector store to retriever
retriever=vectorstore.as_retriever(
    search_kwarg={"k":3}
)
retriever

VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001C435892870>, search_kwargs={})

In [31]:
system_prompt = """You are an assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, just say that you don't know.
Use three sentences maximum and keep the answer concise.

Context: {context}"""

prompt=ChatPromptTemplate.from_messages([
    ("system",system_prompt),
    ("human","{input}")
])
prompt

ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [32]:
### Create a document chain
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableLambda(format_docs)
}), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
| ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nIf you don't know the answer, just say that you don't know.\nUse three sentences maximum and keep the answer concise.\n\nContext: {context}"), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])
| ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.8.4', 'langchain': '1.4.0', 'langchain-google-genai': '4.4.0'}}, 

This Chain:
- Takes retrieved documents.
- "Stuffs" them into prompt's {context} placeholder
- Sends the complete prompt tp LLM
- Returns the LLM's response

In [33]:
### Create the final RAG chain
rag_chain=create_retrieval_chain(retriever,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.chroma.Chroma object at 0x000001C435892870>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks.\nUse the following pieces of retrieved context to answer the question.\nI

In [35]:
response=rag_chain.invoke({"input":"what is deep learning?"})

In [37]:
response["answer"]

'Deep Learning is a specialized branch of machine learning that uses multi-layer artificial neural networks to learn hierarchical representations directly from raw, unstructured data. Rather than relying on manual feature extraction, it automatically discovers abstractions across successive layers using backpropagation and gradient-based optimization. Key architectures include convolutional neural networks, recurrent networks, and transformers, which power breakthroughs in fields like computer vision, speech recognition, and autonomous systems.'